# 🏠 Week 2 Homework 1 of 2 · Completion practice with OpenRouter

**First move: File → Save a copy in Drive, then work in YOUR copy.**

This is practice, at your own pace, on your own time. The shape: three worked examples with the FULL solution visible (read them, run them, change them), then five practice tasks that are yours, then a quick quiz. Full solutions for the tasks wait at the bottom behind the usual try-first warning.

One thing to know before you start: **LLM answers vary between runs.** So the checks in this notebook test the *structure* of what you built (right type, right keys, right length); the quality of the words is judged by your own eyes. Both matter in real systems, and only one of them can be asserted.

## 🔑 Step 0 · Your key, step by step

Same OpenRouter key as class. If you already have it, skip to the code cell. If not:

1. Open **[openrouter.ai](https://openrouter.ai)** in a new tab and sign in (Google login works).
2. Avatar → **Keys** → **Create Key**, name it `buildrlabs`.
3. Copy the key that appears. It looks like `sk-or-...`, shown once. Treat it like cash.
4. With no credits you can use any model whose id ends in **`:free`**: that is our lane tonight. (A `:free` model refusing with a privacy error? Settings → enable free model publication.)

Then run the cell below and paste it at the invisible prompt. Rules from class still apply: the key never goes in a cell, never in git.

**When the runtime resets** (it will, Colab is temporary), your key is gone with it: just re-run the key cell and paste again.

**Set it once, never paste again:** the key cell below checks Colab's Secrets panel first, automatically. Click the 🔑 icon in the left sidebar, add a secret named `OPENROUTER_API_KEY`, paste your key there once, toggle "Notebook access" on, and every session finds it by itself. Running on your laptop instead? The same cell also checks a local `.env` file. Only when both doors are shut does it ask you to paste.

In [5]:
"""
🔑 KEY SETUP · run me first, every session. Three doors, in order:
Colab Secrets -> local .env file -> paste at an invisible prompt.
"""

!pip install -q openai

import os


def load_openrouter_key():
    """Try each key location in order; use the first that answers."""
    try:
        from google.colab import userdata
        value = userdata.get("OPENROUTER_API_KEY")
        if value:
            print("Key loaded from Colab Secrets.")
            return value
    except Exception:
        pass
    try:
        with open(".env") as f:
            for line in f.read().splitlines():
                if line.startswith("OPENROUTER_API_KEY="):
                    value = line.split("=", 1)[1].strip()
                    if value:
                        print("Key loaded from the local .env file.")
                        return value
    except FileNotFoundError:
        pass
    from getpass import getpass
    print("No stored key found: paste it below (it will not display).")
    return getpass("OpenRouter API key: ")


os.environ["OPENROUTER_API_KEY"] = load_openrouter_key()
print("Key is in the environment. Not in the code, not in git.")

No stored key found: paste it below (it will not display).
Key is in the environment. Not in the code, not in git.


In [6]:
# 🧰 THE TOOLKIT · two helpers you will use all notebook. Run me.

import json
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

# Configuration, not code. ":free" = usable with zero credits; current
# list at openrouter.ai/models (filter: free).
MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

def ask(prompt):
    """Send one prompt, return the model's text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


def ask_json(prompt):
    """Send one prompt, parse the reply as JSON, or report failure."""
    reply = ask(prompt)
    reply = reply.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        return {"error": "unparseable", "raw": reply}

print(ask("Say 'toolkit ready' and nothing else."))

toolkit ready


## 📖 Worked example 1 · Steering the answer

A completion does what the prompt says, and only what the prompt says. Watch the same question with three different instructions. Full solution, run it and read the three answers side by side.

In [7]:
# WORKED EXAMPLE 1 · same question, three instructions. Run me.

question = "Why do shops in Colombo close for Poya days?"

plain = ask(question)
short = ask("Answer in exactly one sentence. " + question)
kid = ask("Answer in one sentence a 7 year old would understand. " + question)

print("PLAIN:\n", plain)
print("\nSHORT:\n", short)
print("\nFOR A KID:\n", kid)

# Lesson: the instruction IS the interface. Nothing about the model
# changed between the three calls, only your words did.

PLAIN:
 Shops in Colombo (and across Sri Lanka) close or operate with reduced hours on **Poya days** primarily due to a combination of **religious/cultural observance, national legislation, and widespread social practice**. Here's a breakdown of the key reasons:

1.  **Religious Significance (Core Reason):**
    *   Poya days are the **Buddhist observance days** marking the full moon each lunar month. They commemorate pivotal events in the life of the Buddha (e.g., his birth, enlightenment, first sermon, passing away) and other significant Buddhist historical events.
    *   For observant Buddhists (the majority ethnic group in Sri Lanka, the Sinhalese), Poya days are dedicated to religious activities: visiting temples (*vat*), observing *Sil* (taking precepts, often the 8 or 10 precepts), listening to sermons, meditation, and performing meritorious deeds (*punna*). Engaging in routine commercial activity or worldly pleasures is generally considered inappropriate and distracting from t

## 📖 Worked example 2 · Summarisation

The bread-and-butter completion job: long text in, short text out, with the length and shape YOU chose. Notice how the constraints ("exactly 3 bullet points, each under 12 words") are stated flatly and precisely. Vague prompts get vague summaries.

In [8]:
# WORKED EXAMPLE 2 · summarise with hard constraints. Run me.

announcement = """Dear valued customers, we wish to inform you that from the
first of next month, our Wellawatte and Nugegoda branches will operate on
extended hours, opening at 8am and closing at 9pm on weekdays. Weekend
hours remain unchanged. Additionally, our home delivery service now covers
Mount Lavinia and Dehiwala, with free delivery for orders above LKR 5,000.
Orders below that amount carry a LKR 400 delivery charge. We thank you for
your continued patronage and look forward to serving you better."""

summary = ask(
    "Summarise this announcement as exactly 3 bullet points, each under "
    "12 words, keeping every number exact:\n\n" + announcement
)
print(summary)

- From next month, Wellawatte & Nugegoda open 8am‑9pm weekdays.  
- Weekend hours remain unchanged at both branches.  
- Delivery Mount Lavinia & Dehiwala: free >LKR 5,000, else LKR 400.


## 📖 Worked example 3 · Extraction into JSON

The class finale, now as a reusable pattern: tell the model the exact keys, demand ONLY JSON, and catch the reply with `ask_json` (which unwraps fences and survives bad replies). This pattern is the seed of every structured AI system you will build here.

In [14]:
# WORKED EXAMPLE 3 · text to structured record. Run me.

review = """bought the 1.8L rice cooker from the galle road shop last month.
cooks well but the lid handle broke after two weeks. for LKR 18,500 i
expected better quality. delivery was fast though, came in 2 days."""

record = ask_json(
    "Read this product review and reply with ONLY a JSON object with "
    "exactly these keys: product (string), price_lkr (number), "
    "complaint (string, short), positive_note (string, short).\n\n"
    + review
)
print(record)

# Structure you can assert on, even though the wording varies:
assert "error" not in record, f"Parse failed: {record}"
assert sorted(record.keys()) == ["complaint", "positive_note", "price_lkr", "product"]
assert record["price_lkr"] == 18500
print("✅ Structured, parsed, verified.")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791072000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3IaiK4Tum45bNfOM07aDWCGJLF7'}

## ✏️ Your turn · five practice tasks

Same rules as class drills: TODO gaps, checks below each. Tasks 1 and 2 check structure loosely; tasks 3 to 5 assert hard on structure. Solutions at the bottom, 15-minute rule applies.

In [15]:
"""
✏️ TASK 1 · The polite reply machine
Nimali's angry email deserves a calm answer. Ask the model to write a
polite reply of exactly 3 sentences that: apologises, mentions order
#4471 exactly once, and promises an update within 2 days.
"""

angry_email = """my order #4471 is TWELVE DAYS late. this is the worst
service i have ever experienced. where is my rice cooker???"""

# TODO: build the prompt and call ask()
reply = ask(
    "Write a polite customer-service reply to this email in exactly 3 "
    "sentences. Apologise, mention order #4471 exactly once, and promise "
    "an update within 2 days. Reply with only the reply text.\n\n"
    + angry_email
)

assert isinstance(reply, str) and len(reply) > 40, "Expected a real reply string."
print(reply)
print()
print("CHECK with your eyes: 3 sentences? apology? #4471 once? 2-day promise?")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791072000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3IaiK4Tum45bNfOM07aDWCGJLF7'}

In [12]:
"""
✏️ TASK 2 · Summary with a mood reading
Summarise the announcement from Worked Example 2 into JSON:
{"summary": one sentence, "tone": "positive" | "neutral" | "negative"}
Use ask_json, name the keys in the prompt, demand ONLY JSON.
"""

# TODO: your call here
result = ask_json(
    "Summarise this announcement. Reply with ONLY a JSON object with "
    'exactly these keys: summary (one sentence), tone ("positive", '
    '"neutral" or "negative"). Keep every number exact in the summary.\n\n' + announcement
)

assert "error" not in result, f"Parse failed: {result}"
assert sorted(result.keys()) == ["summary", "tone"], f"Got keys: {sorted(result.keys())}"
assert result["tone"] in ["positive", "neutral", "negative"]
print(result)
print("✅ Task 2 structure verified. Read the summary: did it keep the numbers?")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791072000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3IaiK4Tum45bNfOM07aDWCGJLF7'}

In [ ]:
"""
✏️ TASK 3 · Sentiment, in bulk
Classify all four reviews in ONE call. Ask for ONLY a JSON list of
four strings, each "positive", "negative" or "neutral", in order.
"""

reviews = [
    "absolutely love it, best purchase this year",
    "broke after one week, total waste of money",
    "it is a kettle. it boils water. fine.",
    "delivery guy was rude and the box was damaged",
]

# TODO: build one prompt containing all four reviews, call ask_json
prompt = (
    "Classify the sentiment of each numbered review. Reply with ONLY a "
    'JSON list of exactly 4 strings, in order, each "positive", '
    '"negative" or "neutral".\n\n'
)
for i in range(len(reviews)):
    prompt = prompt + f"{i + 1}. {reviews[i]}\n"

sentiments = ask_json(prompt)

assert isinstance(sentiments, list), f"Expected a list, got: {sentiments}"
assert len(sentiments) == 4
for s in sentiments:
    assert s in ["positive", "negative", "neutral"], f"Unexpected label: {s}"
print(sentiments)
print("✅ Task 3 passed: one call, four classifications.")

In [ ]:
"""
✏️ TASK 4 · Invoice extraction
Pull a typed record out of this invoice text.
Target keys: vendor (string), total_lkr (number), city (string).
"""

invoice = """LANKA APPLIANCES (PVT) LTD, 214 Galle Road, Colombo 03.
Invoice INV-2291. 1x blender LKR 9,500, 1x kettle LKR 3,250.
TOTAL: LKR 12,750. Payment due within 14 days."""

# TODO: ask_json with the three keys named in the prompt
record = ask_json(
    "Extract from this invoice. Reply with ONLY a JSON object with "
    "exactly these keys: vendor (string), total_lkr (number, the grand "
    "total), city (string).\n\n" + invoice
)

assert "error" not in record, f"Parse failed: {record}"
assert sorted(record.keys()) == ["city", "total_lkr", "vendor"]
assert record["total_lkr"] == 12750, f"Got total: {record['total_lkr']}"
print(record)
print("✅ Task 4 passed: paper to data in one call.")

In [ ]:
"""
✏️ TASK 5 · The mini-boss: batch parsing
Three fresh emails. Loop over them, parse EACH into the class contract
{"name": ..., "issue": ..., "urgency": ...} using ask_json, and collect
the three dicts into results. You are building parser v1's engine room.
"""

emails = [
    "hello, chamari gunasekara here. the blender arrived cracked. please replace it urgently.",
    "hi this is ruwan. just checking when the colombo 07 branch opens. no hurry.",
    "i am s. thevarajah. i was charged twice for order #9982. refund the extra payment immediately.",
]

results = []
# TODO: your loop here
for email in emails:
    parsed = ask_json(
        "Read this customer email and reply with ONLY a JSON object with "
        "exactly these keys: name, issue, urgency "
        '(urgency is "high", "medium" or "low").\n\n' + email
    )
    results.append(parsed)


assert len(results) == 3
for r in results:
    assert "error" not in r, f"Parse failed on one email: {r}"
    assert sorted(r.keys()) == ["issue", "name", "urgency"], f"Got keys: {sorted(r.keys())}"
print(json.dumps(results, indent=2))
print("✅ Task 5 passed. Compare this with your v0: how many rules did you NOT write today?")

## 🚢 Ship it

File → Download → Download .ipynb. Then on your repo page: **Add file → Upload files**, drop the notebook in, set the path by editing the name to `week02/homework1_completions.ipynb`, commit message `Week 2 HW1: completion practice, 5 tasks passing`, Commit. (Uploading works for any file, notebooks included.)

Done early? Change the reviews, break the invoice, write a nastier email. The best homework is the version where you tried to make it fail.

## 🔑 Solutions

**Try for 15 minutes per task before you peek.** These need your key cell run first.

In [ ]:
# SOLUTION · Task 1
angry_email = """my order #4471 is TWELVE DAYS late. this is the worst
service i have ever experienced. where is my rice cooker???"""

reply = ask(
    "Write a polite customer-service reply to this email in exactly 3 "
    "sentences. Apologise, mention order #4471 exactly once, and promise "
    "an update within 2 days. Reply with only the reply text.\n\n"
    + angry_email
)
print(reply)

In [ ]:
# SOLUTION · Task 2
announcement = """Dear valued customers, we wish to inform you that from the
first of next month, our Wellawatte and Nugegoda branches will operate on
extended hours, opening at 8am and closing at 9pm on weekdays. Weekend
hours remain unchanged. Additionally, our home delivery service now covers
Mount Lavinia and Dehiwala, with free delivery for orders above LKR 5,000.
Orders below that amount carry a LKR 400 delivery charge. We thank you for
your continued patronage and look forward to serving you better."""

result = ask_json(
    "Summarise this announcement. Reply with ONLY a JSON object with "
    'exactly these keys: summary (one sentence), tone ("positive", '
    '"neutral" or "negative").\n\n' + announcement
)
print(result)

In [ ]:
# SOLUTION · Task 3
reviews = [
    "absolutely love it, best purchase this year",
    "broke after one week, total waste of money",
    "it is a kettle. it boils water. fine.",
    "delivery guy was rude and the box was damaged",
]

prompt = (
    "Classify the sentiment of each numbered review. Reply with ONLY a "
    'JSON list of exactly 4 strings, in order, each "positive", '
    '"negative" or "neutral".\n\n'
)
for i in range(len(reviews)):
    prompt = prompt + f"{i + 1}. {reviews[i]}\n"

sentiments = ask_json(prompt)
print(sentiments)

In [ ]:
# SOLUTION · Task 4
invoice = """LANKA APPLIANCES (PVT) LTD, 214 Galle Road, Colombo 03.
Invoice INV-2291. 1x blender LKR 9,500, 1x kettle LKR 3,250.
TOTAL: LKR 12,750. Payment due within 14 days."""

record = ask_json(
    "Extract from this invoice. Reply with ONLY a JSON object with "
    "exactly these keys: vendor (string), total_lkr (number, the grand "
    "total), city (string).\n\n" + invoice
)
print(record)

In [ ]:
# SOLUTION · Task 5
emails = [
    "hello, chamari gunasekara here. the blender arrived cracked. please replace it urgently.",
    "hi this is ruwan. just checking when the colombo 07 branch opens. no hurry.",
    "i am s. thevarajah. i was charged twice for order #9982. refund the extra payment immediately.",
]

results = []
for email in emails:
    parsed = ask_json(
        "Read this customer email and reply with ONLY a JSON object with "
        "exactly these keys: name, issue, urgency "
        '(urgency is "high", "medium" or "low").\n\n' + email
    )
    results.append(parsed)

print(json.dumps(results, indent=2))